In [1]:
import importlib, env
importlib.reload(env)
from env import CloudClusterEnv, STEPS_PER_WEEK, MIN_PODS, MAX_PODS

import json
import numpy as np
import torch
import torch.nn as nn
import random
from collections import deque

stats = json.load(open('trace_params.json'))['stats']

# ---- discrete action set: DQN can only pick from these ----
# maps action index → change in VMs
DISCRETE_ACTIONS = [-5, -2, 0, +2, +5]   # 5 discrete choices
N_ACTIONS = len(DISCRETE_ACTIONS)

# ---- the Q-network: state → value for each discrete action ----
class DQN(nn.Module):
    def __init__(self, state_dim=32, n_actions=N_ACTIONS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Linear(256, n_actions),          # one Q-value per action
        )
    def forward(self, state):
        return self.net(state)

# quick test
q = DQN()
dummy = torch.zeros(1, 32)
q_values = q(dummy)
print("Q-values shape:", q_values.shape, "(should be [1, 5])")
print("Q-values:", q_values.detach().numpy().round(3))
print("Best action index:", q_values.argmax().item(),
      "→ VM change:", DISCRETE_ACTIONS[q_values.argmax().item()])

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Setup complete. Steps per week: 672
CloudClusterEnv defined.
Q-values shape: torch.Size([1, 5]) (should be [1, 5])
Q-values: [[ 0.078 -0.074 -0.01  -0.074 -0.045]]
Best action index: 0 → VM change: -5


In [2]:
# ---- replay buffer: stores (state, action, reward, next_state, done) ----
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d):
        self.buffer.append((s, a, r, ns, d))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (torch.tensor(np.array(s), dtype=torch.float32),
                torch.tensor(a, dtype=torch.long),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(np.array(ns), dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32))
    def __len__(self):
        return len(self.buffer)

def train_dqn(total_steps=250_000, batch_size=64, gamma=0.99,
              lr=1e-3, target_update=1000,
              eps_start=1.0, eps_end=0.05, eps_decay=50_000):
    env = CloudClusterEnv(stats, seed=None)
    q_net = DQN()
    target_net = DQN()
    target_net.load_state_dict(q_net.state_dict())   # start identical
    optimizer = torch.optim.Adam(q_net.parameters(), lr=lr)
    buffer = ReplayBuffer()

    state, _ = env.reset()
    steps_done = 0
    episode_reward = 0
    episode_rewards = []

    while steps_done < total_steps:
        # epsilon-greedy action selection
        eps = max(eps_end, eps_start - (eps_start - eps_end) * steps_done / eps_decay)
        if random.random() < eps:
            action_idx = random.randrange(N_ACTIONS)      # explore
        else:
            with torch.no_grad():
                qv = q_net(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
                action_idx = qv.argmax().item()           # exploit

        # convert discrete choice to the env's continuous action format
        vm_change = DISCRETE_ACTIONS[action_idx]
        env_action = np.array([vm_change / 5.0])          # env expects [-1,1]*5

        next_state, reward, done, tr, info = env.step(env_action)
        buffer.push(state, action_idx, reward, next_state, float(done))
        episode_reward += reward
        state = next_state
        steps_done += 1

        if done:
            episode_rewards.append(episode_reward)
            episode_reward = 0
            state, _ = env.reset()

        # learn from a batch
        if len(buffer) >= batch_size:
            s, a, r, ns, d = buffer.sample(batch_size)
            q_vals = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                max_next_q = target_net(ns).max(1)[0]
                target = r + gamma * max_next_q * (1 - d)
            loss = nn.functional.mse_loss(q_vals, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # periodically sync target network
        if steps_done % target_update == 0:
            target_net.load_state_dict(q_net.state_dict())

        if steps_done % 10000 == 0:
            recent = np.mean(episode_rewards[-5:]) if episode_rewards else float('nan')
            print(f"steps {steps_done:6d} | eps {eps:.2f} | recent reward {recent:8.1f}")

    return q_net, episode_rewards

print("DQN training function defined.")

DQN training function defined.


In [3]:
# train the DQN baseline (~15-20 min)
print("Training DQN baseline (250k steps)...\n")
dqn_net, dqn_rewards = train_dqn(total_steps=250_000)

torch.save(dqn_net.state_dict(), 'dqn_baseline.pth')
print("\nSaved dqn_baseline.pth")
if len(dqn_rewards) >= 2:
    print(f"First episode reward: {dqn_rewards[0]:.1f}")
    print(f"Last episode reward:  {dqn_rewards[-1]:.1f}")

Training DQN baseline (250k steps)...

steps  10000 | eps 0.81 | recent reward   -112.9
steps  20000 | eps 0.62 | recent reward    -67.9
steps  30000 | eps 0.43 | recent reward    -38.8
steps  40000 | eps 0.24 | recent reward    -24.1
steps  50000 | eps 0.05 | recent reward    -17.1
steps  60000 | eps 0.05 | recent reward    -18.9
steps  70000 | eps 0.05 | recent reward    -18.6
steps  80000 | eps 0.05 | recent reward    -17.2
steps  90000 | eps 0.05 | recent reward    -17.7
steps 100000 | eps 0.05 | recent reward    -15.5
steps 110000 | eps 0.05 | recent reward    -18.6
steps 120000 | eps 0.05 | recent reward    -15.3
steps 130000 | eps 0.05 | recent reward    -18.4
steps 140000 | eps 0.05 | recent reward    -17.4
steps 150000 | eps 0.05 | recent reward    -19.2
steps 160000 | eps 0.05 | recent reward    -18.1
steps 170000 | eps 0.05 | recent reward    -15.3
steps 180000 | eps 0.05 | recent reward    -16.8
steps 190000 | eps 0.05 | recent reward    -16.0
steps 200000 | eps 0.05 | rece

In [4]:
def eval_dqn(q_net, n_episodes=5):
    """Evaluate the trained DQN (greedy, no exploration) on the same env."""
    results = []
    for _ in range(n_episodes):
        env = CloudClusterEnv(stats, seed=None)
        state, _ = env.reset()
        total_cost = total_breaches = total_util = 0.0
        vm_counts = []
        steps = 0
        for t in range(STEPS_PER_WEEK):
            with torch.no_grad():
                qv = q_net(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
                action_idx = qv.argmax().item()          # greedy
            vm_change = DISCRETE_ACTIONS[action_idx]
            env_action = np.array([vm_change / 5.0])
            state, reward, done, tr, info = env.step(env_action)
            total_cost += info['cost']
            total_breaches += info['breaches']
            total_util += info['utilisation']
            vm_counts.append(info['active_vms'])
            steps += 1
            if done:
                break
        results.append({
            'cost': total_cost, 'breaches': total_breaches,
            'util': total_util/steps, 'vms': np.mean(vm_counts)
        })
    return results

# evaluate DQN
print("Evaluating DQN...")
dqn_results = eval_dqn(dqn_net, n_episodes=5)
dqn_cost   = np.mean([r['cost'] for r in dqn_results])
dqn_breach = np.mean([r['breaches'] for r in dqn_results])
dqn_util   = np.mean([r['util'] for r in dqn_results])
dqn_vms    = np.mean([r['vms'] for r in dqn_results])

print("\n" + "="*58)
print("THREE-WAY COMPARISON (same environment, same objective)")
print("="*58)
print(f"{'METRIC':<16}{'HPA':>13}{'DQN':>13}{'PPO':>13}")
print("-"*58)
# using your locked numbers for HPA and PPO (balanced-sla config)
print(f"{'Total cost':<16}{292:>13.0f}{dqn_cost:>13.0f}{184:>13.0f}")
print(f"{'SLA breaches':<16}{1750:>13.0f}{dqn_breach:>13.0f}{427:>13.0f}")
print(f"{'Avg VMs':<16}{8.7:>13.1f}{dqn_vms:>13.1f}{5.5:>13.1f}")
print("="*58)
print(f"\n(PPO = your best sla-focused config; HPA = rule-based baseline)")

Evaluating DQN...

THREE-WAY COMPARISON (same environment, same objective)
METRIC                    HPA          DQN          PPO
----------------------------------------------------------
Total cost                292          173          184
SLA breaches             1750         3082          427
Avg VMs                   8.7          5.2          5.5

(PPO = your best sla-focused config; HPA = rule-based baseline)
